# Word-Level Feature Extraction — Colab T4 GPU

**Goal:** Extract WavLM+prosody features for all videos with labels
**Auto-resume:** Checkpoint saves to Google Drive after every 10 videos
**Runtime:** ~80s per video on T4 GPU

**IMPORTANT:** Run this cell FIRST to see checkpoint status.

In [ ]:
# Cell 0: Setup + Mount Drive + Checkpoint
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, ast, time, subprocess
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/standup4ai'
WORK = '/content/word_level'
FEAT_DIR = f'{BASE}/word_level_features'
CKPT_FILE = f'{BASE}/word_level_checkpoint.json'

os.makedirs(WORK, exist_ok=True)
os.makedirs(FEAT_DIR, exist_ok=True)

# Load checkpoint
done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f))

print(f'Features dir: {FEAT_DIR}')
print(f'Already done: {len(done)} videos')

# Find ALL videos with labels (all languages)
all_videos = []
for lang in ['en_uk', 'en_us', 'es', 'es_latam', 'fr', 'fr_ca', 'it', 'cs', 'hu', 'pt']:
    label_dir = f'{BASE}/seq-Standup4AI/dataset/{lang}/emnlp+jahak/all'
    if os.path.exists(label_dir):
        for f in os.listdir(label_dir):
            if f.endswith('.csv'):
                vid = f.replace('.csv', '')
                if vid not in done:
                    all_videos.append((vid, lang))

print(f'Remaining: {len(all_videos)} videos')

# Show first 10
print(f'Sample: {[v[0] for v in all_videos[:10]]}')

In [ ]:
# Cell 1: Load models (GPU)
import torch
from transformers import AutoModel
import librosa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SR_WAVLM = 16000
SR_PROSODY = 22050

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM ready')

In [ ]:
# Cell 2: Feature extractors
def prosody23(y, sr):
    f = []
    try:
        f0, vd, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]; v = vd[~np.isnan(f0)]
        f.extend([np.mean(f0c) if len(f0c)>0 else 0,
            np.std(f0c) if len(f0c)>0 else 0,
            np.max(f0c) if len(f0c)>0 else 0,
            np.min(f0c) if len(f0c)>0 else 0,
            np.mean(v) if len(v)>0 else 0])
    except: f.extend([0]*5)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    dur = len(y)/sr
    f.extend([dur, dur/(np.sum(rms>np.mean(rms))+1)])
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except: f.extend([0]*5)
    try:
        yh, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except: f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def word_features(y16, y22, t0, t1):
    if t1 - t0 < 0.005: return None
    s16, e = int(t0*SR_WAVLM), min(int(t1*SR_WAVLM), len(y16))
    chunk = y16[s16:e]
    if len(chunk) < int(0.01*SR_WAVLM): return None
    target = 5 * SR_WAVLM
    chunk = np.pad(chunk, (0, max(0, target-len(chunk)))) if len(chunk) < target else chunk[:target]
    with torch.no_grad():
        wemb = wavlm(torch.tensor(chunk/32768.0, dtype=torch.float32).unsqueeze(0).to(device))
        wemb = wemb.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    s22, e = int(t0*SR_PROSODY), min(int(t1*SR_PROSODY), len(y22))
    pros = prosody23(y22[s22:e], SR_PROSODY)
    return np.concatenate([wemb, pros])

In [ ]:
# Cell 3: Process all remaining videos
from tqdm.notebook import tqdm

errors = []
processed = 0
start_time = time.time()

for i, (vid, lang) in enumerate(tqdm(all_videos)):
    feat_path = f'{FEAT_DIR}/{vid}_features.npy'
    
    # Skip if already done
    if os.path.exists(feat_path):
        done.add(vid)
        continue
    
    # Find audio
    audio_path = None
    for folder in [f'{BASE}/audio', f'{BASE}/audio_1000']:
        for ext in ['.m4a', '.wav', '.webm']:
            p = f'{folder}/{vid}{ext}'
            if os.path.exists(p):
                audio_path = p; break
        if audio_path: break
    
    # Find label
    label_path = f'{BASE}/seq-Standup4AI/dataset/{lang}/emnlp+jahak/all/{vid}.csv'
    
    if not audio_path or not os.path.exists(label_path):
        errors.append((vid, 'missing audio or label'))
        continue
    
    try:
        y22, _ = librosa.load(audio_path, sr=SR_PROSODY, mono=True)
        y16, _ = librosa.load(audio_path, sr=SR_WAVLM, mono=True)
        df = pd.read_csv(label_path)
        
        feats, lbls, ts_list = [], [], []
        for _, row in df.iterrows():
            try:
                ts = ast.literal_eval(str(row['timestamp']))
                t0, t1 = float(ts[0]), float(ts[1])
                feat = word_features(y16, y22, t0, t1)
                lbl = str(row.get('label', 'O')).strip()
                if feat is not None:
                    feats.append(feat)
                    lbls.append(1 if lbl in ('B','I','L') else 0)
                    ts_list.append((float(t0), float(t1)))
            except: pass
        
        if feats:
            np.save(feat_path, np.array(feats, dtype=np.float32))
            np.save(f'{FEAT_DIR}/{vid}_labels.npy', np.array(lbls, dtype=np.int32))
            np.save(f'{FEAT_DIR}/{vid}_timestamps.npy', np.array(ts_list, dtype=np.float32))
            done.add(vid)
            processed += 1
            
            # Save checkpoint every 10 videos
            if processed % 10 == 0:
                with open(CKPT_FILE, 'w') as f:
                    json.dump(sorted(done), f)
                elapsed = time.time() - start_time
                rate = processed / elapsed * 3600
                remaining = len(all_videos) - i - 1
                eta = remaining / rate if rate > 0 else 0
                print(f'  Checkpoint: {processed} done, ETA: {eta/3600:.1f}h, rate: {rate:.0f}/h')
        
        del y16, y22
        import gc; gc.collect()
        torch.cuda.empty_cache()
        
    except Exception as e:
        errors.append((vid, str(e)[:100]))
        print(f'  ERROR {vid}: {e}')
        continue

# Final checkpoint
with open(CKPT_FILE, 'w') as f:
    json.dump(sorted(done), f)

elapsed = time.time() - start_time
print(f'\n=== COMPLETE ===')
print(f'Processed: {processed}/{len(all_videos)}')
print(f'Total done: {len(done)}')
print(f'Time: {elapsed/3600:.1f}h')
print(f'Errors: {len(errors)} -> {errors[:5]}')